In [1]:
# ========== 基础工具 ==========
def rc(seq: str) -> str:
    """反向互补"""
    table = str.maketrans('ACGTUacgtu', 'TGCAAtgcaa')
    return seq.translate(table)[::-1]


def read_fasta(path):
    """假设FASTA严格规范：'>'标题行 + 下一行序列"""
    records = []
    with open(path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 2):
        name = lines[i][1:]   # 去掉 '>'
        seq = lines[i + 1]
        records.append((name, seq))
    return records


# ========== 主流程 ==========
def simple_exact_lr_match(ref_fasta, target_fasta, out_txt, unmatched_fasta, use_rc_right=False):
    """
    ref_fasta:       参考FASTA（每条25nt，含名字）
    target_fasta:    待匹配FASTA
    out_txt:         命中输出 (readId_leftRef_rightRef)
    unmatched_fasta: 未匹配输出 (原序列)
    use_rc_right:    是否对右端取反向互补后再匹配参考
    """
    # 1) 建立参考字典
    ref_fwd = {seq: name for name, seq in read_fasta(ref_fasta) if len(seq) == 25}
    ref_rc  = {rc(seq): name for name, seq in ref_fwd.items()}

    # 2) 扫描目标序列
    total = hit = unmatch = 0
    with open(out_txt, 'w') as wtxt, open(unmatched_fasta, 'w') as wfa:
        for name, seq in read_fasta(target_fasta):
            total += 1
            if len(seq) != 50:
                unmatch += 1
                wfa.write(f">{name}\n{seq}\n")
                continue

            left25  = seq[:25]
            right25 = seq[-25:]

            # 左端匹配
            Lname = ref_fwd.get(left25)
            # 右端匹配
            if use_rc_right:
                Rname = ref_rc.get(rc(right25))
            else:
                Rname = ref_fwd.get(right25)

            # 双端匹配才输出
            if Lname and Rname:
                hit += 1
                wtxt.write(f"{name}_{Lname}_{Rname}\n")
            else:
                unmatch += 1
                wfa.write(f">{name}\n{seq}\n")

    print(f"完成：总计 {total} 条；左右都匹配 {hit} 条；未匹配 {unmatch} 条。")




In [2]:

# ========== 批量运行示例 ==========
sample_list = ['1a', '1b', '1s', '2a', '2b', '2s', '3a', '3b', '3s']
part_path   = 'D:/CL/AFISH1/afish_data_20251009/part_data/'
match_path  = 'D:/CL/AFISH1/afish_data_20251009/match_data/'
input_path  = 'D:/CL/AFISH1/code_20251009/input/'

for sample in sample_list:
    ref_fasta     = input_path + "1535_region_list.txt"
    target_fasta  = part_path + sample + "_part_full_info.fasta"
    out_txt       = match_path + sample + "_part_full_info_simple_exact.txt"
    unmatched_fa  = match_path + sample + "_unmatched.fa"

    simple_exact_lr_match(ref_fasta, target_fasta, out_txt, unmatched_fa)

完成：总计 47779761 条；左右都匹配 41585610 条；未匹配 6194151 条。
完成：总计 35948693 条；左右都匹配 31228915 条；未匹配 4719778 条。
完成：总计 17513489 条；左右都匹配 15210812 条；未匹配 2302677 条。
完成：总计 19596195 条；左右都匹配 17010927 条；未匹配 2585268 条。
完成：总计 18075726 条；左右都匹配 15698632 条；未匹配 2377094 条。
完成：总计 18359331 条；左右都匹配 15915049 条；未匹配 2444282 条。
完成：总计 25517925 条；左右都匹配 22087374 条；未匹配 3430551 条。
完成：总计 22178628 条；左右都匹配 19256569 条；未匹配 2922059 条。
完成：总计 22142745 条；左右都匹配 19216009 条；未匹配 2926736 条。
